# Ingest into tables

This notebook reads the raw data downloaded and ingests it into tables in bronze

In [0]:
!ls /Volumes/bronze/methylation/geo_datasets

In [0]:
import pandas as pd



## Ingest Disease data

In [0]:
%sql

DROP TABLE IF EXISTS bronze.methylation.GSE289137_beta_long

In [0]:
disease_raw = pd.read_csv("/Volumes/bronze/methylation/geo_datasets/GSE289137_BetaValues.csv.gz")


In [0]:
disease_raw

In [0]:
import pandas as pd
import os

In [0]:


# Step 2: Melt into long format
df_long = pd.melt(
    disease_raw,
    id_vars=["probe_id"],
    var_name="sample_id",
    value_name="beta"
)





In [0]:
# Step 3: Write in chunks to DBFS as CSV (to avoid driver OOM)
csv_dir = "/Volumes/bronze/methylation/geo_datasets/GSE289137/GSE289137_long_csv"
os.makedirs(csv_dir, exist_ok=True)

In [0]:

#chunk_size = 5_000_000  # adjust if needed
#for i, start in enumerate(range(0, len(df_long), chunk_size)):
#    chunk = df_long.iloc[start:start + chunk_size]
#    chunk.to_csv(f"{csv_dir}/chunk_{i}.csv", index=False)#

#print(f"Wrote CSV chunks to: {csv_dir}")

In [0]:
# Step 4: Read the CSV chunks as a single Spark DataFrame
df_spark = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"dbfs:{csv_dir}")

# Step 5: Save as Spark SQL table
df_spark.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze.methylation.GSE289137_beta_long")

print("✅ Table saved: bronze.methylation.GSE289137_beta_long")


In [0]:
%sql

SELECT * FROM bronze.methylation.GSE289137_beta_long LIMIT 10

### Add variance and restrictions

In [0]:
df = spark.table("bronze.methylation.GSE289137_beta_long")  # or your actual database.table name


In [0]:
from pyspark.sql.functions import lit

df = df.withColumn("access", lit("public"))
# Rename cpg_id to sample_id
#df = df.withColumnRenamed("cpg_id", "sample_id")


In [0]:
from pyspark.sql.functions import variance

# Compute variance of beta_value per sample (cpg_id)
var_df = df.groupBy("probe_id").agg(variance("beta").alias("probe_var"))

# Join back to original table
df = df.join(var_df, on="probe_id", how="left")


In [0]:
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("bronze.methylation.GSE289137_beta_long")


In [0]:
%sql

--DROP TABLE IF EXISTS bronze.methylation.GSE289137_beta_long_with_variance;

SELECT * FROM bronze.methylation.GSE289137_beta_long LIMIT 10

In [0]:
#healthy_raw = pd.read_csv("/Volumes/bronze/methylation/geo_datasets/GSE213478/GPL21145_MethylationEPIC_15073387_v-1-0.csv.gz")
#healthy_raw
